In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 63. Week 43 — Fixed-income measure and control boundary

## 学習目標


- par yield、zero rate、discount factor、forward rateの変換を観測方程式と分ける。
- Treasury daily par yieldをzero-coupon curveとして扱わない。
- finite-horizon dynamic programmingをCoreとし、HJM calibrationやRLをAdvancedへ置く。


## 前提知識


- B1のcash-flow discountingとB2のconditional expectation
- B7のdynamic term-structure、B4のfinite-horizon optimization

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 63


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Measure vocabulary

Treasury snapshotのpar_yieldはcoupon cash flowを同一yieldで価格付けするpar quoteである。以下のzero/forwardは、変換の既知の形を検査するための数学fixtureであり、Treasury par quoteをzero rateへ読み替えたものではない。

$$
D(t)=e^{-t z(t)},\qquad f(t_i,t_{i+1})=-\frac{t_{i+1}z(t_{i+1})-t_i z(t_i)}{t_{i+1}-t_i}
$$

In [4]:
fixture_times = np.array([0.25, 1.0, 2.0, 5.0, 10.0])
fixture_zero = np.array([0.045, 0.043, 0.041, 0.039, 0.040])
discount = np.exp(-fixture_times * fixture_zero)
forward = -np.diff(fixture_times * fixture_zero) / np.diff(fixture_times)
measure_table = pd.DataFrame(
    {
        "maturity_years": fixture_times,
        "illustrative_zero_rate": fixture_zero,
        "discount_factor": discount,
        "treasury_latest_par_yield": np.interp(fixture_times, maturity_years, curve_yields[-1]),
    }
)
display(measure_table)
assert np.all((discount > 0.0) & (discount <= 1.0))
fig = go.Figure()
fig.add_scatter(x=fixture_times, y=100.0 * fixture_zero, mode="lines+markers", name="zero fixture")
fig.add_scatter(x=fixture_times[1:], y=100.0 * forward, mode="lines+markers", name="forward fixture")
fig.add_scatter(x=maturity_years, y=curve_yields[-1], mode="lines+markers", name="Treasury par quote")
fig.update_layout(title="Par quote versus illustrative zero/forward measures", xaxis_title="Years", yaxis_title="Percent", template="plotly_white")
fig.show()

,maturity_years,illustrative_zero_rate,discount_factor,treasury_latest_par_yield
0,0.25,0.045,0.988813,3.670000
1,1.00,0.043,0.957911,3.584286
2,2.00,0.041,0.921272,3.470000
3,5.00,0.039,0.822835,3.730000
4,10.00,0.040,0.670320,4.180000


In [5]:
transition = np.array(
    [
        [[0.80, 0.20], [0.25, 0.75]],
        [[0.55, 0.45], [0.10, 0.90]],
    ]
)
stage_cost = np.array([[0.20, 0.45], [0.55, 0.10]])
terminal_cost = np.array([0.30, 0.80])
control = qt.finite_horizon_control(transition, stage_cost, terminal_cost, horizon=5)
assert control.values.shape == (6, 2)
control_table = pd.DataFrame(control.values, columns=["state_0_value", "state_1_value"])
display(control_table)
fig = go.Figure()
fig.add_scatter(x=np.arange(control.values.shape[0]), y=control.values[:, 0], mode="lines+markers", name="state 0")
fig.add_scatter(x=np.arange(control.values.shape[0]), y=control.values[:, 1], mode="lines+markers", name="state 1")
fig.update_layout(title="Finite-horizon backward induction values", xaxis_title="Remaining stage index", yaxis_title="Expected cost", template="plotly_white")
fig.show()

,state_0_value,state_1_value
0,1.42885,1.235575
1,1.25550,1.122250
2,1.06500,1.017500
3,0.85000,0.925000
4,0.60000,0.850000
5,0.30000,0.800000


## 2. 失敗モード

- par yieldをdiscount factorへ直接代入する。
- forward curveの符号規約を定義しない。
- solver successをno-arbitrage証明と呼ぶ。
- HJMのno-arbitrage restrictionと実証calibrationを一つのfitへ混ぜる。

## 3. 段階別演習

### 基礎

1. coupon cash flowからpar quoteがどのdiscount factor制約を満たすか導出せよ。

### 標準

2. fixture zeroを変え、discount monotonicityとnegative-rateケースを比較せよ。

### 研究

3. HJM drift restrictionを仮定とデータ要件に分解し、Coreへ入れない理由を書け。

## 4. Exit Criteria

- [ ] par/zero/discount/forwardを定義した
- [ ] Treasury par quoteをzero rateへ変換していない
- [ ] finite-horizon recursionを解析的に検算した
- [ ] pricing measureとforecasting measureを区別した

## 5. 出典

- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)
- [U.S. Treasury Daily Treasury Par Yield Curve Rates](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve)

- [Fama and MacBeth (1973), Risk, Return, and Equilibrium](https://www.jstor.org/stable/1831028)
- [Hansen (1982), Large Sample Properties of GMM Estimators](https://doi.org/10.2307/1912775)
- [Duffie and Kan (1996), A Yield-Factor Model of Interest Rates](https://doi.org/10.1016/0304-405X(95)00881-6)
- [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf)